# Watch MVCC Bloat and Vacuum in a Real Postgres

Companion notebook for the [Vacuum and Autovacuum](https://codemia.io/courses/postgresql_deep_dive/pg_vacuum_and_autovacuum) lesson on Codemia.

**What you will build**: a set of experiments on a live PostgreSQL server. You will look inside a heap page with `pageinspect` and watch an update leave the old row version behind, measure how a table grows when nothing cleans up, see what plain `VACUUM` and `VACUUM FULL` each give back, pin the cleanup horizon with one idle transaction and watch vacuum refuse to remove anything, and compare HOT updates across fillfactor settings.

Runs on Colab with no GPU. Uses PostgreSQL, `psycopg2`, `numpy` and `matplotlib`.

## Setup

On Colab the next cell installs PostgreSQL with apt and starts it, which takes about a minute. Anywhere else, point the `PG_DSN` environment variable at a server you can create tables on, for example a Docker container started with `docker run -e POSTGRES_PASSWORD=postgres -p 5432:5432 postgres:14`.

In [ ]:
%pip install -q psycopg2-binary numpy matplotlib

In [ ]:
import os, sys, time, random, threading, subprocess
import psycopg2
import psycopg2.errors
import numpy as np
import matplotlib.pyplot as plt

if "google.colab" in sys.modules:
    def sh(cmd):
        subprocess.run(cmd, shell=True, check=True, stdout=subprocess.DEVNULL)
    sh("apt-get -qq update && apt-get -qq install -y postgresql postgresql-contrib")
    sh("service postgresql start")
    sh("sudo -u postgres psql -qc \"ALTER USER postgres PASSWORD 'postgres'\"")

DSN = os.environ.get("PG_DSN", "host=localhost user=postgres password=postgres dbname=postgres")

def connect():
    conn = psycopg2.connect(DSN)
    conn.autocommit = True
    return conn

db = connect()

def q(sql, args=None, conn=None):
    with (conn or db).cursor() as cur:
        cur.execute(sql, args)
        return cur.fetchall() if cur.description else None

print(q("SELECT version()")[0][0])

## 1. One row, three versions

Insert a row and update it twice. `heap_page_items` shows every tuple on the page, including the versions no query can see any more. `t_xmin` is the transaction that created a version, `t_xmax` the one that replaced it, and `t_ctid` points from each old version to its successor.

In [ ]:
q("CREATE EXTENSION IF NOT EXISTS pageinspect")
q("CREATE EXTENSION IF NOT EXISTS pgstattuple")
q("DROP TABLE IF EXISTS demo")
q("CREATE TABLE demo (id int PRIMARY KEY, v int) WITH (autovacuum_enabled = off)")
q("INSERT INTO demo VALUES (1, 10)")
q("UPDATE demo SET v = 11 WHERE id = 1")
q("UPDATE demo SET v = 12 WHERE id = 1")

print("lp  xmin    xmax    ctid")
for lp, xmin, xmax, ctid in q("SELECT lp, t_xmin, t_xmax, t_ctid FROM heap_page_items(get_raw_page('demo', 0))"):
    print("%-3s %-7s %-7s %s" % (lp, xmin, xmax, ctid))
print("\nwhat a query sees:", q("SELECT ctid, xmin, v FROM demo"))

Three physical tuples, one visible row. The first two have a nonzero `xmax`: they are dead once no running transaction could still need them, but they keep occupying the page until something removes them.

## 2. Bloat when nothing cleans up

Build a 100,000 row table with autovacuum switched off for it, then update every row a few times. `pgstattuple` scans the table and reports exact live and dead tuple counts and free space, unlike the statistics views, which are estimates updated in the background.

In [ ]:
def make_table(name, fillfactor=100, rows=100_000):
    q(f"DROP TABLE IF EXISTS {name}")
    q(f"CREATE TABLE {name} (id int PRIMARY KEY, balance int, note text) "
      f"WITH (autovacuum_enabled = off, fillfactor = {fillfactor})")
    q(f"INSERT INTO {name} SELECT g, 0, repeat('x', 80) FROM generate_series(1, {rows}) g")
    q(f"VACUUM ANALYZE {name}")

def tuple_stats(name):
    size, live, dead, free = q(f"SELECT table_len, tuple_count, dead_tuple_count, free_space FROM pgstattuple('{name}')")[0]
    return dict(mb=size / 2**20, live=live, dead=dead, free_mb=free / 2**20)

def scan_ms(name, reps=5):
    best = float("inf")
    for _ in range(reps):
        t = time.perf_counter()
        q(f"SELECT sum(balance) FROM {name}")
        best = min(best, (time.perf_counter() - t) * 1000)
    return best

def show(label, name):
    s = tuple_stats(name)
    print("%-24s size %6.1f MB  live %7d  dead %7d  free %5.1f MB  scan %6.1f ms"
          % (label, s["mb"], s["live"], s["dead"], s["free_mb"], scan_ms(name)))
    return s

make_table("accounts")
history = [("fresh", show("fresh", "accounts"))]
for i in range(1, 4):
    q("UPDATE accounts SET balance = balance + 1")
    history.append((f"update {i}", show(f"after update {i}", "accounts")))

Every full table update writes a new version of every row. The table keeps growing even though the number of live rows never changes, and a sequential scan has to read all of it. Growth is somewhat less than one table size per round because Postgres prunes dead versions opportunistically when it visits a nearly full page, but pruning alone does not keep up.

## 3. What VACUUM and VACUUM FULL give back

Plain `VACUUM` marks dead space reusable but rarely shrinks the file. Later updates fill that space instead of extending the table. `VACUUM FULL` rewrites the table compactly and returns the space to the operating system, at the price of an `ACCESS EXCLUSIVE` lock for the whole rewrite.

In [ ]:
q("VACUUM accounts")
history.append(("vacuum", show("after VACUUM", "accounts")))
q("UPDATE accounts SET balance = balance + 1")
history.append(("update 4", show("update after vacuum", "accounts")))
q("VACUUM FULL accounts")
history.append(("vacuum full", show("after VACUUM FULL", "accounts")))

labels = [h[0] for h in history]
fig, ax = plt.subplots(figsize=(8, 3.5))
live_mb = [h[1]["mb"] - h[1]["free_mb"] for h in history]
ax.bar(labels, live_mb, label="used")
ax.bar(labels, [h[1]["free_mb"] for h in history], bottom=live_mb, label="free (reusable)")
ax.set_ylabel("table size, MB"); ax.legend(); ax.set_title("Updates grow the table; VACUUM makes space reusable; VACUUM FULL shrinks it")
plt.xticks(rotation=20); plt.tight_layout(); plt.show()

## 4. One idle transaction pins the horizon

Vacuum may only remove a dead version when no transaction could still see it. Open a second connection, start a `REPEATABLE READ` transaction and read the table once, then leave it open, as a forgotten session or a long report would. Update every row and vacuum.

In [ ]:
make_table("accounts")
reader = psycopg2.connect(DSN)
reader.set_session(isolation_level="REPEATABLE READ")
q("SELECT count(*) FROM accounts", conn=reader)

q("UPDATE accounts SET balance = balance + 1")
db.notices.clear()
q("VACUUM (VERBOSE) accounts")
print("".join(n for n in db.notices if "dead" in n or "removable" in n).strip())
print()
show("vacuum with reader open", "accounts")
print("reader backend_xmin:", q("SELECT backend_xmin, now() - xact_start FROM pg_stat_activity WHERE backend_xmin IS NOT NULL AND pid <> pg_backend_pid()"))

reader.commit(); reader.close()
q("VACUUM accounts")
show("vacuum after commit", "accounts");

With the reader open, vacuum reports 100,000 versions that are dead but not yet removable, and nothing is reclaimed. The moment the transaction ends, the same vacuum removes them all. In production the culprits are sessions left `idle in transaction`, long analytics queries on the primary, a standby with `hot_standby_feedback`, and abandoned replication slots. `idle_in_transaction_session_timeout` and monitoring the age of `backend_xmin` catch the first two.

## 5. HOT updates and fillfactor

An update that changes no indexed column and fits on the same page is HOT: no index entries are written, and pruning can later reclaim the old version without vacuum. A lower fillfactor leaves free space on every page for exactly this. Update one fifth of the rows five times on tables with different fillfactors and count HOT updates.

In [ ]:
def hot_stats(name, expected_updates, timeout=10):
    deadline = time.time() + timeout
    while True:
        q("SELECT pg_stat_clear_snapshot()")
        upd, hot = q("SELECT n_tup_upd, n_tup_hot_upd FROM pg_stat_user_tables WHERE relname = %s", (name,))[0]
        if upd >= expected_updates or time.time() > deadline:
            return upd, hot
        time.sleep(0.25)

results = []
for ff in (100, 90, 70, 50):
    name = f"ff{ff}"
    make_table(name, fillfactor=ff)
    q("SELECT pg_stat_reset_single_table_counters(%s::regclass)", (name,))
    for r in range(5):
        q(f"UPDATE {name} SET balance = balance + 1 WHERE id % 5 = {r}")
    upd, hot = hot_stats(name, 100_000)
    s = tuple_stats(name)
    results.append((ff, s["mb"], 100 * hot / max(upd, 1)))
    print("fillfactor %3d  size %5.1f MB  HOT %5.1f%% of %d updates" % (ff, s["mb"], 100 * hot / max(upd, 1), upd))

fig, ax1 = plt.subplots(figsize=(6.5, 3.5))
ffs = [str(r[0]) for r in results]
ax1.bar(ffs, [r[2] for r in results], color="tab:green"); ax1.set_ylabel("HOT updates, percent"); ax1.set_xlabel("fillfactor")
ax2 = ax1.twinx(); ax2.plot(ffs, [r[1] for r in results], color="black", marker="o"); ax2.set_ylabel("table size after updates, MB")
plt.title("Free space on the page turns updates into HOT updates"); plt.tight_layout(); plt.show()

At fillfactor 100 the first updates find full pages and move rows elsewhere. Pruning then frees space that later updates reuse, so the HOT share rises over time but stays partial. With reserved space most updates are HOT from the start. The cost is a larger table up front, and the benefit is fewer index writes and less vacuum work. That trade pays off for tables updated often on columns without indexes.

Now add an index on the updated column and repeat once: HOT drops to zero, because every new version needs its own index entry.

In [ ]:
make_table("ffidx", fillfactor=70)
q("CREATE INDEX ON ffidx (balance)")
q("SELECT pg_stat_reset_single_table_counters('ffidx'::regclass)")
q("UPDATE ffidx SET balance = balance + 1 WHERE id % 5 = 0")
upd, hot = hot_stats("ffidx", 20_000)
print("indexed column updated: %d updates, %d HOT" % (upd, hot))

## 6. Exercises

1. Turn autovacuum back on for `accounts` with `ALTER TABLE accounts SET (autovacuum_enabled = on, autovacuum_vacuum_scale_factor = 0.01)`, run updates in a loop for a minute and watch `n_dead_tup` and `last_autovacuum` in `pg_stat_user_tables`.
2. Repeat section 4 with a `READ COMMITTED` reader that is idle between statements but still inside a transaction. Does it still pin the horizon? What about a session that is idle outside any transaction?
3. Measure index bloat: run `pgstatindex` on the primary key after section 2 and after `REINDEX INDEX CONCURRENTLY`.